<a href="https://colab.research.google.com/github/Harshita1204/python/blob/main/CaseStudy_medicalDiagnosis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CNN(convolutional neural network) -> detects tumor present or not -> gives out 95% accuracy
# but in medical terms prediction is not enough , we need reasoning , evidence, interpretbility .
# wrong prediction= wrong treatment

# Grad-CAM
# it answers which paret of the image made the model says 'tumor'
# CNN -> looks at the MRI image: it sees patterns, it extracts features , it makes a decision
# but grad-cam let us peek inside the model's brain

# output of Grad-CAM : Red -> important region , blue-> not important
# Heatmap shows -> skull boundary(edges) instead of tumor region


How grad-CAM actually works:
1.CNN processes image -> produces feature maps(patterns)
2. pick output class-> example"tumor"
3. backpropogation -> compute gradients -> which feature affected this prediction?
4. weight feature map-> important features get higher weight
5. combine them -> create a heatMap



In [1]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt


In [18]:
# what this does is : takes your CNN model , a target layer (usually last conv layer)
# sets hooks to capture: gradients, activations
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.gradients = None
        self.activations = None

        target_layer.register_backward_hook(self.save_gradient)
        target_layer.register_forward_hook(self.save_activation)

    # save gradients
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    # stores gradients during backpropagations

    # save activations
    def save_activation(self, module, input, output):
        self.activations = output
        # stores feature maps (important patterns detected)

    # generate heatmaps
    def generate(self, input_image, class_idx):
        output = self.model(input_image)
        # forward pass = prediction

        self.model.zero_grad()
        loss = output[0, class_idx]
        loss.backward()
        #backward pass:
        #focus only on selected class(tumor)
        # compute gradients
        gradients = self.gradients[0].cpu().data.numpy()
        activations = self.activations[0].cpu().data.numpy()
        # converts tensors-> numpy

        # compute importance
        weights = np.mean(gradients, axis=(1, 2))
        # each feature map gets a weight
        # higher weight= more important

        # build a heatmap
        cam = np.zeros(activations.shape[1:], dtype=np.float32)

        for i, w in enumerate(weights):
            cam += w * activations[i]
            # combine feature maps* importance

        cam = np.maximum(cam, 0)
        # remove negative values(ReLU)

        cam = cv2.resize(cam, (224, 224))
        cam = cam - np.min(cam)
        cam = cam / np.max(cam)
        # normalize heatmaps

        # Convert input_image to a displayable format (numpy array, 0-255, BGR if using cv2)
        # Assuming input_image is a PyTorch tensor of shape (1, C, H, W) and normalized
        img_np = input_image.squeeze(0).permute(1, 2, 0).cpu().numpy()
        # Denormalize if necessary (e.g., if it was normalized to [0,1])
        # For example, if normalized with mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]:
        # img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = (img_np * 255).astype(np.uint8) # Scale to 0-255
        img_np = cv2.resize(img_np, (224, 224)) # Resize to match heatmap

        # Convert image to BGR for OpenCV if it's RGB
        if img_np.shape[2] == 3 and img_np.ndim == 3:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

        # overlay
        heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
        overlay = heatmap * 0.4 + img_np * 0.6 # Use img_np instead of img

        return overlay, heatmap # It's good practice to return the overlay and heatmap
